<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkST0151ENSkillsNetwork20531532-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>



# **Final Project: Boston Housing**


##### Estimated time needed: **60** minutes


#### Import the required libraries we need for the lab.


In [ ]:
import piplite
await piplite.install(['numpy'],['pandas'])
await piplite.install(['seaborn'])

In [ ]:
import pandas as pd
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as pyplot
import scipy.stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

#### Read the dataset in the csv file from the URL


In [ ]:
from js import fetch
import io

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ST0151EN-SkillsNetwork/labs/boston_housing.csv'
resp = await fetch(URL)
boston_url = io.BytesIO((await resp.arrayBuffer()).to_py())

In [ ]:
boston_df=pd.read_csv(boston_url)

#### Add your code below following the instructions given in the course to complete the peer graded assignment


In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from scipy import stats
import statsmodels.api as sm

In [ ]:
df = pd.read_csv("boston_housing.csv")
df.head()

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
df.duplicated().sum()

In [ ]:
corr = df.corr(numeric_only=True)
corr

In [ ]:
plt.figure(figsize=(10,8))
plt.imshow(corr, aspect='auto')
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Correlation Matrix (Pearson)")
plt.tight_layout()
plt.show()

In [ ]:
##Task 2: Descriptive Statistics and Visualizations
df.describe()

In [ ]:
##Boxplot for MEDV
plt.figure(figsize=(6,4))
plt.boxplot(df["MEDV"].dropna(), vert=True)
plt.title("MEDV Boxplot")
plt.ylabel("MEDV (Median value of owner-occupied homes)")
plt.show()

In [ ]:
##Bar plot for CHAS
chas_counts = df["CHAS"].value_counts().sort_index()
plt.figure(figsize=(5,4))
plt.bar(chas_counts.index.astype(str), chas_counts.values)
plt.title("Count of Homes by Charles River Boundary (CHAS)")
plt.xlabel("CHAS (0=No, 1=Yes)")
plt.ylabel("Count")
plt.show()

chas_counts

In [ ]:
##Discretize AGE into 3 groups + Boxplot MEDV vs AGE groups
##Create AGE groups
# AGE = proportion of owner-occupied units built prior to 1940
# Common cut strategy: terciles (equal-sized groups)
df["AGE_GROUP"] = pd.qcut(df["AGE"], q=3, labels=["Low AGE", "Medium AGE", "High AGE"])
df[["AGE", "AGE_GROUP"]].head()

In [ ]:
##Boxplot MEDV by AGE_GROUP
groups = [df.loc[df["AGE_GROUP"] == g, "MEDV"].dropna() for g in ["Low AGE", "Medium AGE", "High AGE"]]

plt.figure(figsize=(7,4))
plt.boxplot(groups, labels=["Low AGE", "Medium AGE", "High AGE"])
plt.title("MEDV by AGE Group (AGE discretized into 3 groups)")
plt.xlabel("AGE Group")
plt.ylabel("MEDV")
plt.show()

In [ ]:
##Scatter plot NOX vs INDUS
##Scatter plot
plt.figure(figsize=(6,4))
plt.scatter(df["INDUS"], df["NOX"], alpha=0.7)
plt.title("NOX vs INDUS")
plt.xlabel("INDUS (non-retail business acres per town)")
plt.ylabel("NOX (nitric oxide concentration)")
plt.show()

In [ ]:
##Histogram for PTRATIO
plt.figure(figsize=(6,4))
plt.hist(df["PTRATIO"].dropna(), bins=15)
plt.title("PTRATIO Histogram")
plt.xlabel("PTRATIO (pupil-teacher ratio)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
##Task 3: Apply Statistical Tests

In [ ]:
##Levene’s test (equal variances) for MEDV by CHAS
medv_chas0 = df.loc[df["CHAS"] == 0, "MEDV"].dropna()
medv_chas1 = df.loc[df["CHAS"] == 1, "MEDV"].dropna()

levene_stat, levene_p = stats.levene(medv_chas0, medv_chas1, center='median')
levene_stat, levene_p

In [ ]:
##T-test: MEDV difference by CHAS
equal_var = (levene_p > 0.05)

t_stat, t_p = stats.ttest_ind(medv_chas0, medv_chas1, equal_var=equal_var)
t_stat, t_p

In [ ]:
##Add group means (management-friendly)
medv_chas0.mean(), medv_chas1.mean(), medv_chas1.mean() - medv_chas0.mean()

In [ ]:
##ANOVA: MEDV among 3 AGE groups
g1 = df.loc[df["AGE_GROUP"] == "Low AGE", "MEDV"].dropna()
g2 = df.loc[df["AGE_GROUP"] == "Medium AGE", "MEDV"].dropna()
g3 = df.loc[df["AGE_GROUP"] == "High AGE", "MEDV"].dropna()

anova_stat, anova_p = stats.f_oneway(g1, g2, g3)
anova_stat, anova_p

In [ ]:
##Pearson correlation test: NOX and INDUS
r, p = stats.pearsonr(df["NOX"].dropna(), df["INDUS"].dropna())
r, p

In [ ]:
##Simple Linear Regression: DIS --> MEDV
X = df["DIS"]
y = df["MEDV"]

X = sm.add_constant(X)   # adds intercept
model = sm.OLS(y, X, missing="drop").fit()
model.summary()

In [ ]:
##Extract key numbers-management-ready
beta = model.params["DIS"]
pval = model.pvalues["DIS"]
r2 = model.rsquared

beta, pval, r2